In [1]:
!nvidia-smi

Tue Mar  3 15:58:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             41W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
!git clone https://github.com/sunpengfei1122/Adaptive_axonal_delay
%cd Adaptive_axonal_delay

!pip install lava-dl h5py tonic

Cloning into 'Adaptive_axonal_delay'...
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 27 (delta 7), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (27/27), 10.79 KiB | 10.79 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/Adaptive_axonal_delay
ERROR: Ignored the following versions that require a different python version: 0.4.0 Requires-Python >=3.8,<3.11; 0.5.0 Requires-Python >=3.8,<3.11; 0.6.0 Requires-Python <3.11,>=3.10
ERROR: Could not find a version that satisfies the requirement lava-dl (from versions: none)
ERROR: No matching distribution found for lava-dl


In [ ]:
import sys
import os
import h5py
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datetime import datetime
import numpy as np
import matplotlib
import yaml
matplotlib.use('Agg')



In [4]:
!wget https://zenkelab.org/datasets/shd_train.h5.gz
!wget https://zenkelab.org/datasets/shd_test.h5.gz
!gunzip shd_train.h5.gz
!gunzip shd_test.h5.gz

--2026-03-03 15:58:27--  https://zenkelab.org/datasets/shd_train.h5.gz
Resolving zenkelab.org (zenkelab.org)... 88.198.116.88
Connecting to zenkelab.org (zenkelab.org)|88.198.116.88|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 130839480 (125M) [application/x-gzip]
Saving to: ‘shd_train.h5.gz’

shd_train.h5.gz     100%[===================>] 124.78M  4.03MB/s    in 18s     

2026-03-03 15:58:46 (6.75 MB/s) - ‘shd_train.h5.gz’ saved [130839480/130839480]

--2026-03-03 15:58:46--  https://zenkelab.org/datasets/shd_test.h5.gz
Resolving zenkelab.org (zenkelab.org)... 88.198.116.88
Connecting to zenkelab.org (zenkelab.org)|88.198.116.88|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 38137408 (36M) [application/x-gzip]
Saving to: ‘shd_test.h5.gz’

shd_test.h5.gz      100%[===================>]  36.37M  18.4MB/s    in 2.0s    

2026-03-03 15:58:49 (18.4 MB/s) - ‘shd_test.h5.gz’ saved [38137408/38137408]



In [5]:


# 2. Check what other files exist in repo
!ls

main  README.md  shd_test.h5  shd_train.h5


In [6]:
!pip install git+https://github.com/bamsumit/slayerPytorch.git

  Cloning https://github.com/bamsumit/slayerPytorch.git to /tmp/pip-req-build-q883khfu
  Running command git clone --filter=blob:none --quiet https://github.com/bamsumit/slayerPytorch.git /tmp/pip-req-build-q883khfu
  Resolved https://github.com/bamsumit/slayerPytorch.git to commit 01beeeb6a181546d6c6830382ce6086bfc587836
  Preparing metadata (setup.py) ... done
ERROR: More than one .egg-info directory found in /tmp/pip-pip-egg-info-pdnmc89d


In [19]:
!find / -name "slayerPytorch" -type d 2>/dev/null


In [20]:
# Go to main slayerPytorch dir
%cd /content/Adaptive_axonal_delay/Adaptive_axonal_delay/slayerPytorch

# Build the CUDA extension
!python setup.py build_ext --inplace

[Errno 2] No such file or directory: '/content/Adaptive_axonal_delay/Adaptive_axonal_delay/slayerPytorch'
/content/Adaptive_axonal_delay
python3: can't open file '/content/Adaptive_axonal_delay/setup.py': [Errno 2] No such file or directory


In [32]:
import sys
sys.path.append('/content/Adaptive_axonal_delay/Adaptive_axonal_delay/slayerPytorch/src')
import slayer as snn
print("slayer loaded!")

ModuleNotFoundError: No module named 'slayer'

In [24]:
!find /content -name "slayerPytorch" -type d 2>/dev/null
!find /content -name "setup.py" 2>/dev/null
!ls /content/
!ls /content/Adaptive_axonal_delay/

Adaptive_axonal_delay  sample_data
data  main  network.yaml  README.md  shd_test.h5  shd_train.h5


In [25]:
# Cell 1 - Clone slayerPytorch into the repo folder
%cd /content/Adaptive_axonal_delay
!git clone https://github.com/bamsumit/slayerPytorch.git

# Cell 2 - Build CUDA extension
%cd /content/Adaptive_axonal_delay/slayerPytorch
!python setup.py build_ext --inplace 2>&1 | tail -3

# Cell 3 - Fix numpy bug
!sed -i 's/from numpy.lib.arraysetops import isin/from numpy import isin/' \
  /content/Adaptive_axonal_delay/slayerPytorch/src/slayerParams.py

# Cell 4 - Import
import sys
sys.path.append('/content/Adaptive_axonal_delay/slayerPytorch')
import src as snn
print("slayer loaded!")

/content/Adaptive_axonal_delay
Cloning into 'slayerPytorch'...
remote: Enumerating objects: 3293, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 3293 (delta 37), reused 29 (delta 29), pack-reused 3250 (from 3)
Receiving objects: 100% (3293/3293), 87.47 MiB | 45.28 MiB/s, done.
Resolving deltas: 100% (1190/1190), done.
/content/Adaptive_axonal_delay/slayerPytorch
x86_64-linux-gnu-g++ -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O2 -Wall -g -fstack-protector-strong -Wformat -Werror=format-security -g -fwrapv -O2 -shared -Wl,-O1 -Wl,-Bsymbolic-functions -Wl,-Bsymbolic-functions -g -fwrapv -O2 build/temp.linux-x86_64-cpython-312/src/cuda/slayerLoihiKernels.o -L/usr/local/lib/python3.12/dist-packages/torch/lib -L/usr/local/cuda/lib64 -L/usr/lib/x86_64-linux-gnu -lc10 -ltorch -ltorch_cpu -ltorch_python -lcudart -lc10_cuda -ltorch_cuda -o build/lib.linux-x86_64-cpython-312/slayerLoihiCuda.cpython-312-x86_64-linux-gnu.so
c

/content/Adaptive_axonal_delay/slayerPytorch/src/spikeLoss.py:44: SyntaxWarning: invalid escape sequence '\i'
  E = \int_0^T \\left( \\varepsilon * (output -desired) \\right)(t)^2\\ \\text{d}t
/content/Adaptive_axonal_delay/slayerPytorch/src/spikeLoss.py:69: SyntaxWarning: invalid escape sequence '\i'
  \\frac{acutalSpikeCount - desiredSpikeCount}{targetRegionLength} & \\text{for }t \in targetRegion\\\\


In [26]:
# Step 1 - Rewrite yaml
yaml_content = """
neuron:
    type:     SRMALPHA
    theta:    10
    tauSr:    10.0
    tauRef:   1.0
    scaleRef: 2
    tauRho:   1
    scaleRho: 1

simulation:
    Ts:         1.0
    tSample:    400
    numSamples: 8156

training:
    error:
        type: NumSpikes
        tgtSpikeRegion:
            start: 0
            stop:  400
        tgtSpikeCount:
            true:  80
            false: 10
"""

with open('network.yaml', 'w') as f:
    f.write(yaml_content)

# Step 2 - Reload netParams fresh
netParams = snn.params('network.yaml')

# Step 3 - Verify it loaded correctly
print(netParams['training']['error']['type'])  # should print NumSpikes

NumSpikes


In [12]:
!cat /content/Adaptive_axonal_delay/Adaptive_axonal_delay/slayerPytorch/src/__init__.py

cat: /content/Adaptive_axonal_delay/Adaptive_axonal_delay/slayerPytorch/src/__init__.py: No such file or directory


In [30]:
# Fix the slayerParams.py file
!sed -i 's/from numpy.lib.arraysetops import isin/from numpy import isin/' \
  /content/Adaptive_axonal_delay/Adaptive_axonal_delay/slayerPytorch/src/slayerParams.py
# Try this instead
import sys
sys.path.append('/content/Adaptive_axonal_delay/Adaptive_axonal_delay/slayerPytorch')

import src as snn
print(dir(snn))

sed: can't read /content/Adaptive_axonal_delay/Adaptive_axonal_delay/slayerPytorch/src/slayerParams.py: No such file or directory
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'io', 'layer', 'learningStats', 'loihi', 'loss', 'optimizer', 'params', 'predict', 'quantize', 'quantizeParams', 'slayer', 'slayerLoihi', 'slayerParams', 'spikeClassifier', 'spikeFileIO', 'spikeLoss', 'utils']


In [27]:
import sys
sys.path.append('/content/Adaptive_axonal_delay/Adaptive_axonal_delay/slayerPytorch')
import src as snn

error = snn.loss(netParams)
print(dir(error))


['T_destination', '__annotations__', '__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_apply', '_backward_hooks', '_backward_pre_hooks', '_buffers', '_call_impl', '_compiled_call_impl', '_forward_hooks', '_forward_hooks_always_called', '_forward_hooks_with_kwargs', '_forward_pre_hooks', '_forward_pre_hooks_with_kwargs', '_get_backward_hooks', '_get_backward_pre_hooks', '_get_name', '_is_full_backward_hook', '_load_from_state_dict', '_load_state_dict_post_hooks', '_load_state_dict_pre_hooks', '_maybe_warn_non_full_backward_hook', '_modules', '_named_members', '_non_persistent_buffers_set', '_parameters', '_register_load_state_dict_pre_hoo

In [36]:

# ── Install slayerPytorch ──────────────────────────────────────────
# Run this in a separate cell first:
# !git clone https://github.com/bamsumit/slayerPytorch.git
# !pip install -e slayerPytorch/
# import slayerSNN as snn

# ── Preprocess H5 → NPY ───────────────────────────────────────────
def preprocess_shd(h5_path, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    with h5py.File(h5_path, 'r') as f:
        spikes_times = f['spikes']['times']
        spikes_units = f['spikes']['units']
        labels = f['labels'][:]
        index_list = []
        for i in range(len(labels)):
            times = np.array(spikes_times[i])
            units = np.array(spikes_units[i])
            # quantize time to 400 bins (0-1 sec → 400 steps)
            times_q = np.clip((times * 400).astype(int), 0, 399)
            spike_array = np.zeros((700, 400), dtype=np.float32)
            spike_array[units, times_q] = 1
            np.save(os.path.join(out_dir, f'{i}.npy'), spike_array)
            index_list.append([i, int(labels[i])])
    np.savetxt(out_dir + '.txt', index_list, fmt='%d')
    print(f"Saved {len(index_list)} samples to {out_dir}")

# Run preprocessing
preprocess_shd('/content/Adaptive_axonal_delay/shd_train.h5', 'data/train')
preprocess_shd('/content/Adaptive_axonal_delay/shd_test.h5',  'data/test')

# ── Dataset ───────────────────────────────────────────────────────
class shdDataset(Dataset):
    def __init__(self, data_paths):
        self.data_paths = data_paths
        self.samples = np.loadtxt(self.data_paths + '.txt').astype('int')

    def __len__(self):
        return self.samples.shape[0]

    def __getitem__(self, index):
        inputindex = self.samples[index, 0]
        label      = self.samples[index, 1]
        path  = self.data_paths + '/' + str(inputindex.item()) + '.npy'
        input = torch.from_numpy(np.load(path))
        label = torch.from_numpy(np.asarray(label))

        emptytensor = torch.zeros((1, 1, 700, 400))
        yy = input.transpose(1, 0).nonzero()
        
        # ✅ Fix: clip column indices to max 399
        row_idx = yy[:, 0]
        col_idx = yy[:, 1].clamp(0, 399)
        
        emptytensor[0, 0, row_idx, col_idx] = 1
        desiredClass = torch.zeros((20, 1, 1, 1))
        desiredClass[label, ...] = 1

        return emptytensor, desiredClass, label

# ── Network ───────────────────────────────────────────────────────
class Network(torch.nn.Module):
    def __init__(self, netParams):
        super(Network, self).__init__()
        slayer = snn.layer(netParams['neuron'], netParams['simulation'])
        self.slayer = slayer
        self.fc1    = torch.nn.utils.weight_norm(slayer.dense(700, 128), name='weight')
        self.fc2    = torch.nn.utils.weight_norm(slayer.dense(128, 128), name='weight')
        self.fc3    = torch.nn.utils.weight_norm(slayer.dense(128, 20),  name='weight')
        self.delay1 = slayer.delay(128)
        self.delay2 = slayer.delay(128)

    def forward(self, spike):
        spike = spike.reshape((spike.shape[0], -1, 1, 1, spike.shape[-1]))
        spike = self.slayer.spike(self.fc1(self.slayer.psp(spike)))
        spike = self.delay1(spike)
        spike = self.slayer.spike(self.fc2(self.slayer.psp(spike)))
        spike = self.delay2(spike)
        spike = self.slayer.spike(self.fc3(self.slayer.psp(spike)))
        return spike

    def clamp(self, thea1, thea2):
        self.delay1.delay.data.clamp_(0, thea1)
        self.delay2.delay.data.clamp_(0, thea2)

# ── Main ──────────────────────────────────────────────────────────
trainedFolder = 'test2'
logsFolder    = 'testL2'
os.makedirs(trainedFolder, exist_ok=True)
os.makedirs(logsFolder,    exist_ok=True)

netParams = snn.params('network.yaml')
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ✅ Fixed: single GPU for Colab
net    = torch.nn.DataParallel(Network(netParams).to(device), device_ids=[0])
module = net.module

error     = snn.loss(netParams).to(device)
optimizer = snn.utils.optim.Nadam(net.parameters(), lr=0.1)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[800], gamma=0.5)

# ✅ Fixed: num_workers=2 for Colab
trainingSet = shdDataset('data/train')
trainLoader = DataLoader(trainingSet, batch_size=128, shuffle=True,  num_workers=2)
testingSet  = shdDataset('data/test')
testLoader  = DataLoader(testingSet,  batch_size=128, shuffle=False, num_workers=2)

stats = snn.utils.stats()
print("Parameters: %.2fM" % sum(p.nelement() for p in module.parameters()))

# ✅ Fixed: initialize update counters
update1 = 0
update2 = 0
thea1   = 64
thea2   = 64

for epoch in range(300):
    tSt = datetime.now()

    for i, (input, target, label) in enumerate(trainLoader):
        net.train()
        input  = input.to(device)
        target = target.to(device)
        output = net(input)

        stats.training.correctSamples += torch.sum(snn.predict.getClass(output) == label).data.item()
        stats.training.numSamples     += len(label)

        loss = error.numSpikes(output, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # ✅ Fixed: clamp1() → clamp(64,64)
        if epoch <= 250:
            module.clamp(64, 64)
        else:
            update1 += 1
            update2 += 1
            for name, parameters in net.named_parameters():
                if name == 'module.delay1.delay' and update1 > 150:
                    sorted1, _ = torch.sort(torch.floor(parameters.detach().squeeze()))
                    thea1 = torch.max(torch.floor(parameters.detach().squeeze()))
                    if sorted1[108] > (thea1 - 5):
                        thea1 += 1
                        update1 = 0
                elif name == 'module.delay2.delay' and update2 > 150:
                    sorted2, _ = torch.sort(torch.floor(parameters.detach().squeeze()))
                    thea2 = torch.max(torch.floor(parameters.detach().squeeze()))
                    if sorted2[108] > (thea2 - 5):
                        thea2 += 1
                        update2 = 0
            module.clamp(thea1=thea1, thea2=thea2)

        stats.training.lossSum += loss.cpu().data.item()
        stats.print(epoch, i)

    for i, (input, target, label) in enumerate(testLoader):
        net.eval()
        with torch.no_grad():
            input  = input.to(device)
            target = target.to(device)
            output = net(input)
            stats.testing.correctSamples += torch.sum(snn.predict.getClass(output) == label).data.item()
            stats.testing.numSamples     += len(label)
            loss = error.numSpikes(output, target)
            stats.testing.lossSum += loss.cpu().data.item()
            stats.print(epoch, i)

    stats.update()
    stats.plot(saveFig=True, path=trainedFolder + '/')
    if stats.testing.bestAccuracy:
        torch.save(module.state_dict(), trainedFolder + '/SHD_best.pt')
    if epoch % 10 == 0:
        torch.save({'net': module.state_dict(), 'optimizer': optimizer.state_dict()},
                   logsFolder + f'/checkpoint{epoch}.pt')
    stats.save(trainedFolder + '/')

Saved 8156 samples to data/train
Saved 2264 samples to data/test
Parameters: 109076.00M

Epoch :          0(i =       0)
loss = 9.9992                             accuracy = 0.078125                 

Epoch :          0(i =       1)
loss = 157.4                              accuracy = 0.066406                 

Epoch :          0(i =       2)
loss = 110.65                             accuracy = 0.059896                 

Epoch :          0(i =       3)
loss = 85.561                             accuracy = 0.054688                 

Epoch :          0(i =       4)
loss = 70.403                             accuracy = 0.051562                 

Epoch :          0(i =       5)
loss = 60.362                             accuracy = 0.048177                 

Epoch :          0(i =       6)
loss = 53.088                             accuracy = 0.047991                 

Epoch :          0(i =       7)
loss = 47.766                             accuracy = 0.045898                 

Epoch :        

In [ ]:
print("Allocated (MB):", torch.cuda.memory_allocated() / 1024**2)
print("Reserved (MB):", torch.cuda.memory_reserved() / 1024**2)

In [15]:
!find /content/Adaptive_axonal_delay -name "*.yaml" -o -name "*.yml" 2>/dev/null

/content/Adaptive_axonal_delay/network.yaml


In [16]:
!cat /content/Adaptive_axonal_delay/Adaptive_axonal_delay/slayerPytorch/network.yaml

cat: /content/Adaptive_axonal_delay/Adaptive_axonal_delay/slayerPytorch/network.yaml: No such file or directory


In [ ]:
!cat /content/Adaptive_axonal_delay/Adaptive_axonal_delay/slayerPytorch/example/02_NMNIST_MLP/network.yaml

simulation:
    Ts: 1.0
    tSample: 300
    nSample: 12
neuron:
    type:     SRMALPHA
    theta:    10
    tauSr:    10.0
    tauRef:   1.0
    scaleRef: 2     # relative to theta
    tauRho:   1     # relative to theta #0.43429448190325176
    scaleRho: 1
training:
    error:
        type: NumSpikes #ProbSpikes #NumSpikes
        probSlidingWin: 20  # only valid for ProbSpikes
        tgtSpikeRegion: {start: 0, stop: 300}    # only valid for NumSpikes and ProbSpikes
        tgtSpikeCount: {true: 60, false: 10}    # only valid for NumSpikes
    path:
        # out:     OutFiles/
        in:      NMNISTsmall/
        train:   NMNISTsmall/train1K.txt
        test:    NMNISTsmall/test100.txt


['T_destination', '__annotations__', '__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_apply', '_backward_hooks', '_backward_pre_hooks', '_buffers', '_call_impl', '_compiled_call_impl', '_forward_hooks', '_forward_hooks_always_called', '_forward_hooks_with_kwargs', '_forward_pre_hooks', '_forward_pre_hooks_with_kwargs', '_get_backward_hooks', '_get_backward_pre_hooks', '_get_name', '_is_full_backward_hook', '_load_from_state_dict', '_load_state_dict_post_hooks', '_load_state_dict_pre_hooks', '_maybe_warn_non_full_backward_hook', '_modules', '_named_members', '_non_persistent_buffers_set', '_parameters', '_register_load_state_dict_pre_hoo